# System Design Framework & Mental Models

> **The mental models, estimation tools, and decision frameworks used across
> all system design problems — from back-of-envelope to CAP theorem.**

**Topics:**
1. The 7-Step Repeatable Design Framework
2. Back-of-Envelope Estimation Calculator
3. CAP Theorem & Consistency Spectrum
4. PACELC — A More Complete Lens
5. The Interview Playbook

## 1 · The 7-Step Repeatable Design Framework

> **Every architectural decision trades one property for another. The senior engineer
> names the trade-off; the junior engineer ignores it.**

```
1. CLARIFY   → Functional requirements, then scale (DAU, QPS, p99 latency, data retention)
               Key questions: read-heavy or write-heavy? Strong or eventual consistency?

2. ESTIMATE  → QPS, storage/year, bandwidth — find the BOTTLENECK
               The bottleneck determines the architecture, not the business logic.

3. API       → Define the contract (REST/gRPC/GraphQL, request/response schema)
               It constrains everything downstream. Wrong API = rewrite the clients.

4. DATA      → Schema, SQL vs NoSQL, access patterns, sharding key
               Wrong data model = rewrite the whole system.

5. HIGH LEVEL→ LB → stateless app → cache → DB + async workers  (the standard shape)
               Draw this first, then identify where it breaks at your estimated QPS.

6. DEEP DIVE → Scale the bottleneck from step 2
               Cache reads? Shard writes? Replicate? CDN? Async queue? Denormalize?

7. OPERATE   → Failures, timeouts, retries, circuit breakers, monitoring, alerting
               Senior engineers spend HALF their design time here.
```

### 🌍 How Top Companies Apply This Framework

| Company | Step-2 Bottleneck | Step-6 Solution |
|---|---|---|
| **Stripe** | Idempotency for payments, not raw throughput | Idempotency keys in DB; not sharding |
| **Twitter** | Fan-out for 50M-follower celebrity tweets | Hybrid: pre-compute normal timelines, pull for celebrities |
| **Netflix** | CDN bandwidth (not DB reads) | 10k+ edge servers cache content per-region |
| **Uber** | Real-time driver location (1M drivers × update every 4s = 250k writes/sec) | Cassandra for write throughput |
| **WhatsApp** | Message delivery guarantee at scale | Persistent TCP connection per device + ack protocol |

---
## 2 · Back-of-Envelope Calculator

In [ ]:
"""Back-of-envelope calculator — run this for any system design estimate."""

def back_of_envelope(
    dau: int,
    actions_per_user_per_day: float,
    avg_payload_bytes: int,
    storage_per_record_bytes: int,
    peak_multiplier: float = 3.0,
    years: int = 5,
):
    total_actions_per_day = dau * actions_per_user_per_day
    avg_rps = total_actions_per_day / 86_400
    peak_rps = avg_rps * peak_multiplier
    bandwidth_gbps = (avg_rps * avg_payload_bytes) / 1e9
    storage_5yr_gb = (total_actions_per_day * 365 * years * storage_per_record_bytes) / 1e9

    print(f"=== Back-of-Envelope Estimate ===")
    print(f"DAU: {dau:,}")
    print(f"Total actions/day: {total_actions_per_day:,.0f}")
    print(f"Average RPS: {avg_rps:,.1f}")
    print(f"Peak RPS ({peak_multiplier}×): {peak_rps:,.1f}")
    print(f"Bandwidth: {bandwidth_gbps:.3f} GB/s")
    print(f"Storage ({years}yr): {storage_5yr_gb:,.1f} GB")

    # Key decision signals
    if peak_rps > 100_000:
        print("⚠️  Sharding likely needed — single DB writer saturates around 50-100k writes/sec")
    if storage_5yr_gb > 10_000:
        print("⚠️  Distributed storage / object store likely needed (>10TB)")
    if bandwidth_gbps > 10:
        print("⚠️  CDN essential — 10+ GB/s origin bandwidth is unsustainable")

# --- Twitter tweet scenario ---
back_of_envelope(
    dau=300_000_000,
    actions_per_user_per_day=0.33,  # ~100M tweets/day
    avg_payload_bytes=500,
    storage_per_record_bytes=1_000,
)
print()
# --- Stripe payments scenario ---
back_of_envelope(
    dau=5_000_000,
    actions_per_user_per_day=1,     # 1 charge/user/day average
    avg_payload_bytes=2_000,
    storage_per_record_bytes=2_000,
)

## 3 · CAP Theorem & Consistency Spectrum

> **During a network partition (cables get cut, data centers lose connectivity — WILL happen),
> you can guarantee EITHER Consistency OR Availability, but not both.
> The real question: which failure mode hurts your business more?**

### CP vs AP — The Practical Choice

```
CP systems (PostgreSQL, HBase, ZooKeeper):
  During partition → return an error. Never serve stale data.
  Business impact  → "service unavailable" — frustrating but CORRECT.
  Use for          → bank balances, order records, distributed locks.

AP systems (Cassandra, DynamoDB, Riak):
  During partition → return potentially stale data. Always available.
  Business impact  → user sees slightly outdated data — usually minor.
  Use for          → shopping carts, product catalogues, user profiles.
```

### The Consistency Spectrum

```
STRONG ←─────────────────────────────────────────────────────→ WEAK

Linearizable  Serializable  Causal  Read-your-writes  Eventual
     |              |          |           |               |
   Highest        ACID       Happens-   User sees        Lowest
   latency       (DB tx)    before    own writes        latency
```

### PACELC — A More Complete Lens

```
IF   Partition:   C or A  ← the CAP choice
ELSE (normal op): L or C  ← latency vs consistency trade-off

DynamoDB:   PA/EL   (Available during partition; low latency in normal operation)
Cassandra:  PA/EL   (same)
PostgreSQL: PC/EC   (Consistent during partition; consistent in normal operation)
HBase:      PC/EC   (same)
```

### Decision Guide

| Data type | Requirement | Choose |
|---|---|---|
| Bank balance | Zero tolerance for stale reads | CP (Postgres, 2PL transactions) |
| Shopping cart | Minor staleness OK | AP (Cassandra, eventual consistency) |
| Product catalogue | 1-minute-old price fine | AP |
| Order record | Must be exactly right | CP |
| Distributed lock | Must be exclusive | CP (ZooKeeper/etcd with Raft) |
| User session | Re-login is acceptable failure | AP |

---
## 5 · The Interview Playbook — Estimations

In [ ]:
"""Back-of-envelope calculator — run this for any system design estimate."""

In [ ]:
def back_of_envelope(
    dau: int,
    actions_per_user_per_day: float,
    avg_payload_bytes: int,
    storage_per_record_bytes: int,
    peak_multiplier: float = 3.0,
    years: int = 5,
):
    total_actions_per_day = dau * actions_per_user_per_day
    avg_rps = total_actions_per_day / 86_400
    peak_rps = avg_rps * peak_multiplier
    bandwidth_gbps = (avg_rps * avg_payload_bytes) / 1e9
    storage_5yr_gb = (total_actions_per_day * 365 * years * storage_per_record_bytes) / 1e9

    print(f"=== Back-of-Envelope Estimate ===")
    print(f"DAU: {dau:,}")
    print(f"Total actions/day: {total_actions_per_day:,.0f}")
    print(f"Average RPS: {avg_rps:,.1f}")
    print(f"Peak RPS ({peak_multiplier}×): {peak_rps:,.1f}")
    print(f"Bandwidth: {bandwidth_gbps:.3f} GB/s")
    print(f"Storage ({years}yr): {storage_5yr_gb:,.1f} GB")

    # Key decision signals
    if peak_rps > 100_000:
        print("⚠️  Sharding likely needed — single DB writer saturates around 50-100k writes/sec")
    if storage_5yr_gb > 10_000:
        print("⚠️  Distributed storage / object store likely needed (>10TB)")
    if bandwidth_gbps > 10:
        print("⚠️  CDN essential — 10+ GB/s origin bandwidth is unsustainable")

In [ ]:
# --- Twitter tweet scenario ---
back_of_envelope(
    dau=300_000_000,
    actions_per_user_per_day=0.33,  # ~100M tweets/day
    avg_payload_bytes=500,
    storage_per_record_bytes=1_000,
)
print()

In [ ]:
# --- Stripe payments scenario ---
back_of_envelope(
    dau=5_000_000,
    actions_per_user_per_day=1,     # 1 charge/user/day average
    avg_payload_bytes=2_000,
    storage_per_record_bytes=2_000,
)

"""
Interview framework cheat sheet — run to print a structured interview guide.
Bring this mental checklist to every system design session.
"""

INTERVIEW_PLAYBOOK = """
╔══════════════════════════════════════════════════════════════════╗
║           SYSTEM DESIGN INTERVIEW PLAYBOOK                      ║
╠══════════════════════════════════════════════════════════════════╣
║ 1. CLARIFY  (5 min)                                             ║
║    □ DAU / peak QPS                                             ║
║    □ Read:write ratio                                           ║
║    □ p99 latency SLA                                            ║
║    □ Consistency: strong or eventual ok?                        ║
║    □ Global or single region?                                   ║
╠══════════════════════════════════════════════════════════════════╣
║ 2. ESTIMATE (3 min)                                             ║
║    □ avg RPS = requests/day ÷ 86,400                            ║
║    □ peak RPS = avg × 3-5×                                      ║
║    □ storage/year = records/day × 365 × record_size             ║
║    □ bandwidth = avg_rps × payload_size                         ║
║    → Identify: is this read-heavy? write-heavy? storage-bound?  ║
╠══════════════════════════════════════════════════════════════════╣
║ 3. API CONTRACT                                                 ║
║    □ Write the key 3-5 endpoints                                ║
║    □ Include: idempotency key for mutations? pagination?        ║
╠══════════════════════════════════════════════════════════════════╣
║ 4. DATA MODEL                                                   ║
║    □ Entities, relationships, access patterns                   ║
║    □ SQL vs NoSQL (justify by access pattern, consistency need) ║
║    □ Shard key (high cardinality, matches queries)              ║
╠══════════════════════════════════════════════════════════════════╣
║ 5. HIGH-LEVEL DESIGN                                            ║
║    Client→CDN→LB→App (stateless)→Cache→DB                      ║
║    + Async queue for slow/spiky work                            ║
╠══════════════════════════════════════════════════════════════════╣
║ 6. DEEP DIVE (bottleneck)                                       ║
║    □ Name the bottleneck from your estimate                     ║
║    □ Apply the right pattern: cache / shard / async / CB / etc  ║
╠══════════════════════════════════════════════════════════════════╣
║ 7. OPERATE IT                                                   ║
║    □ Monitoring: rate, latency, errors (golden signals)         ║
║    □ Failure modes: what if cache/DB/queue goes down?           ║
║    □ Deployment: canary, blue-green                             ║
║    □ RPO / RTO: backup and recovery                             ║
╠══════════════════════════════════════════════════════════════════╣
║ RED FLAGS to avoid:                                             ║
║  ✗ Jumping to microservices without justification              ║
║  ✗ No idempotency on mutation endpoints                         ║
║  ✗ Ignoring failure modes                                       ║
║  ✗ Using Kafka without explaining the consumer                  ║
║  ✗ Sharding before replication                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(INTERVIEW_PLAYBOOK)